# We made a chatbot earlier but that dumb bot could not even remember my name!
## Lets make a better chatbot with memory

In [1]:
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage, SystemMessage
from typing import TypedDict, List
import os
from dotenv import load_dotenv
from uuid import uuid4

In [2]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=api_key)

In [3]:
class State(TypedDict):
    messages: List[BaseMessage]
    user_input: str

In [4]:
def chatbot_node(state: State):
    current_messages = state.get("messages", [])
    user_input = state.get("user_input", "")
    if not current_messages:
        current_messages = [
            SystemMessage(content="You are a helpful AI assistant. You have memory of our entire conversation and can refer back to previous messages and details shared.")
        ]
    current_messages.append(HumanMessage(content=user_input))
    prompt = ChatPromptTemplate.from_messages([
        MessagesPlaceholder("messages"),
    ])
    chain = prompt | llm
    response = chain.invoke({"messages": current_messages})
    current_messages.append(response)
    return {"messages": current_messages, "user_input": ""}

In [5]:
from uuid import uuid4
thread_id = uuid4()
print(thread_id)
config = {"configurable": {"thread_id": str(thread_id)}}

f9e1440f-1b3c-4610-a3ee-2d23e897d867


In [6]:
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot_node)
graph_builder.set_entry_point("chatbot")
graph_builder.add_edge("chatbot", END)

In [7]:
graph = graph_builder.compile(checkpointer=MemorySaver())

Generally, the product is accessed by multiple people at a time, so its better to use random UUID as thread_id. but even basic 1,2,3,4... would work.

In [8]:
thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

In [11]:
print("Explicit Memory Chatbot Ready!")
print("Commands: '/exit' to quit, '/history' to see conversation, '/debug' for state info")
print("-" * 50)
while True:
    user_input = input("You: ").strip()
    if user_input.lower() == "/exit":
        print("Goodbye!")
        break
    elif user_input.lower() == "/history":
        try:
            state = graph.get_state(config)
            messages = state.values.get("messages", [])
            print(f"\n--- Conversation History ({len(messages)} messages) ---")
            for i, msg in enumerate(messages, 1):
                if isinstance(msg, SystemMessage):
                    print(f"{i}. [SYSTEM]: {msg.content}")
                elif isinstance(msg, HumanMessage):
                    print(f"{i}. You: {msg.content}")
                elif isinstance(msg, AIMessage):
                    print(f"{i}. Bot: {msg.content}")
            print("--- End History ---\n")
        except Exception as e:
            print(f"Error: {e}")
        continue
    
    elif user_input.lower() == "/debug":
        try:
            state = graph.get_state(config)
            print(f"\nDebug Info:")
            print(f"Thread ID: {thread_id}")
            print(f"Messages in state: {len(state.values.get('messages', []))}")
            print(f"State keys: {list(state.values.keys())}")
        except Exception as e:
            print(f"Debug error: {e}")
        continue
    
    if not user_input:
        continue
    
    try:
        result = graph.invoke(
            {"user_input": user_input}, 
            config=config
        )
        messages = result.get("messages", [])
        if messages:
            last_msg = messages[-1]
            if isinstance(last_msg, AIMessage):
                print(f"Bot: {last_msg.content}")
        
    except Exception as e:
        print(f"Error: {e}")
        print(f"Error type: {type(e)}")

print("Session ended.")

Explicit Memory Chatbot Ready!
Commands: '/exit' to quit, '/history' to see conversation, '/debug' for state info
--------------------------------------------------


You:  Hey my name is Pratirath, I like F1


Bot: Nice to meet you, Pratirath. F1 fan, eh? Which team and driver do you support? Are you a fan of the current champion or do you have a soft spot for a particular team with a rich history like Ferrari or McLaren?


You:  Whats my name?


Bot: Your name is Pratirath. You mentioned it in our very first message. We were just discussing your interest in Formula 1 (F1).


...


--- Conversation History (9 messages) ---
1. [SYSTEM]: You are a helpful AI assistant. You have memory of our entire conversation and can refer back to previous messages and details shared.
2. You: Hey my name is Pratirath, I like F1
3. Bot: Nice to meet you, Pratirath. F1 fan, eh? Which team and driver do you support? Are you a fan of the current champion or do you have a soft spot for a particular team with a rich history like Ferrari or McLaren?
4. You: Whats my name?
5. Bot: Your name is Pratirath. You mentioned it in our very first message. We were just discussing your interest in Formula 1 (F1).
6. You: Yeah, I also like Football
7. Bot: You're a multi-sport fan, I like that. Football and F1, quite different but both exciting in their own ways. Do you have a favorite football team or league, like the English Premier League or La Liga?
8. You: Tell me about my interests
9. Bot: You have two main interests that we've discussed so far: 

1. **Formula 1 (F1)**: You mentioned you lik

You:  /debug



Debug Info:
Thread ID: 89b26a8d-2ea2-4307-9c47-aeaa0968604b
Messages in state: 9
State keys: ['messages', 'user_input']


You:  /exit


Goodbye!
Session ended.


## Much better Chatbot! Atleast not so dumb.